In [17]:
import numpy as np
import pandas as pd 
import torch
from itertools import combinations
from collections import defaultdict

In [62]:
pd.set_option('display.max_columns', None) 
articles = pd.read_csv("data/articles.csv")
articles.head(5)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,111565001,111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
2,111586001,111586,Shape Up 30 den 1p Tights,273,Leggings/Tights,Garment Lower body,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Tights with built-in support to lift the botto...
3,111593001,111593,Support 40 den 1p Tights,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny tights that shape the tummy, thighs..."
4,111609001,111609,200 den 1p Tights,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Opaque matt tights. 200 denier.


In [19]:
print(f"articles columns: {articles.columns}")

articles columns: Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')


In [66]:
articles['product_group_name'].unique()

array(['Garment Upper body', 'Socks & Tights', 'Garment Lower body',
       'Accessories', 'Items', 'Underwear', 'Unknown', 'Nightwear',
       'Swimwear', 'Shoes', 'Garment Full body', 'Bags',
       'Underwear/nightwear', 'Garment and Shoe care', 'Cosmetic',
       'Stationery'], dtype=object)

In [20]:
cus = pd.read_csv("data/customers.csv")
trans = pd.read_csv("data/transactions_train.csv")

In [21]:
cus.head(2)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...
1,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,1.0,1.0,ACTIVE,Regularly,56.0,b31984b20a8c478de38eaf113c581ff64e63c4242e607b...


In [22]:
trans.head(2)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2020-08-09,0004df3751c43c51f79aa24d321f35ce4a1a3ec180c4a1...,832482001,0.016932,1
1,2020-08-09,0004df3751c43c51f79aa24d321f35ce4a1a3ec180c4a1...,802881001,0.067780,1


In [65]:
# Kiểm tra số lượng giao dịch theo ngày
daily_transactions = trans.groupby("t_dat").size().reset_index(name="transaction_count")
print("Số lượng giao dịch theo ngày:")
print(daily_transactions)
print(f"\nTổng ngày: {len(daily_transactions)}")
print(f"Trung bình giao dịch/ngày: {daily_transactions['transaction_count'].mean():.2f}")

Số lượng giao dịch theo ngày:
        t_dat  transaction_count
0  2020-08-09              26888
1  2020-08-10              36716
2  2020-08-11              42557
3  2020-08-12              70712
4  2020-08-13              39175
5  2020-08-14              35192
6  2020-08-15              29288

Tổng ngày: 7
Trung bình giao dịch/ngày: 40075.43


In [63]:
trans.tail(2)

,t_dat,customer_id,article_id,price,sales_channel_id
280526,2020-08-15,fff9c6f12f40bf233de2d1fb398541247495f764d1975a...,0779725006,0.016932,2
280527,2020-08-15,fff9c6f12f40bf233de2d1fb398541247495f764d1975a...,0779725006,0.016932,2


In [64]:
len(trans)

280528

In [53]:
articles["article_id"] = articles["article_id"].astype(str).str.zfill(10)
trans["article_id"] = trans["article_id"].astype(str).str.zfill(10)

In [54]:
# nhóm item theo userr
user_groups = trans.groupby("customer_id")["article_id"].apply(list)

edge_counter = defaultdict(int)

for items in user_groups:
    items = list(set(items))  # remove duplicate
    for a, b in combinations(items, 2):
        edge_counter[(a, b)] += 1
        edge_counter[(b, a)] += 1

edges = list(edge_counter.keys())
weights = list(edge_counter.values())

In [55]:
article_ids = list(set(trans["article_id"]))
id2idx = {aid: i for i, aid in enumerate(article_ids)}

edge_index = torch.tensor([
    [id2idx[a], id2idx[b]] for a, b in edges
]).t()

edge_weight = torch.tensor(weights, dtype=torch.float)

In [56]:
clip_data = torch.load("clip_embeddings.pt")  
# {article_id: embedding}

X = torch.zeros(len(article_ids), 512)

for aid, idx in id2idx.items():
    X[idx] = clip_data[aid]

In [57]:
import torch.nn as nn
from torch_geometric.nn import SAGEConv

class HGNN(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=128, out_dim=64):
        super().__init__()
        
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return x  # [num_nodes, 64]

f:\ptit\anam4_ki2\RecommenderSystem\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [58]:
import torch.nn.functional as F

def contrastive_loss(emb, edge_index, num_nodes, num_neg=1):
    src, dst = edge_index

    # positive distance
    pos_dist = ((emb[src] - emb[dst])**2).sum(dim=1)

    # negative sampling
    neg_src = torch.randint(0, num_nodes, (len(src),))
    neg_dst = torch.randint(0, num_nodes, (len(src),))

    neg_dist = ((emb[neg_src] - emb[neg_dst])**2).sum(dim=1)

    loss = pos_dist.mean() - neg_dist.mean()
    return loss